
# Hi Eng. Yahia 

# I'm Renad/Rnad Jqaim and here my final project, I hope that all steps are clear...

# Thanks

****

# Please read this before start : 

each step were numbered like the final projecr mlops docuement 
here https://docs.google.com/document/d/1u4NZKdes1im8nUVxJ6RBfo5ZA9tUXysl-6MKdKqWUx4/edit?tab=t.0#heading=h.9hbyeml0w464
****

# STEP 1 : Choose datasets
We used health dataset
here is the dataset name and URL we are using .. 

Dataset : Sleep Health and Lifestyle Dataset


https://www.kaggle.com/datasets/uom190346a/sleep-health-and-lifestyle-dataset

In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uom190346a/sleep-health-and-lifestyle-dataset")

print("Path to dataset files:", path)

# Step 2: Frame the Business Problem
Objective/goal :: To determine how lifestyle behaviors (such as stress level, physical activity, sleep duration) and job-related factors (like occupation) influence the Quality of Sleep in individuals.

Why This Is Important:::

Poor sleep affects mental and physical health, productivity, and emotional stability..  bby identifying key lifestyle or occupational patterns that lead to poor sleep .. wellness programs and individuals can make informed decisions to improve sleep habits.

importance :::: 

Sleep quality affects mental health, productivity, and overall well-being
Type of Problem: Classification (since "Quality of Sleep" is a categorical label)

Chosen Metricss ::::
- Accuracy: to measure overall correct predictions ..
- F1-Score: Because some sleep quality levels might be less frequent, and we care about catching them ........

# Step 3 : Load Data & Undserstand it

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/sleep-health-and-lifestyle-dataset/Sleep_health_and_lifestyle_dataset.csv')

df.head()


In [ ]:

#understand the data 
#show the missing data 
#show the target 

print("\nDataset Info:")
print(df.info())
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Distribution:\n", df['Quality of Sleep'].value_counts())

# my target is :: Quality of Sleep

In [ ]:
# model importing and ML libraites
# 📦 Import libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ml libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC


In [ ]:
# enocde categorties var.....
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# Step 4 : SPlit data

In [ ]:
### split steep
# sseparate features and target
X = df.drop(columns=['Quality of Sleep'])   # target 
y = df['Quality of Sleep']           #### featurwe
print (X) 
print ("------------------------------")
print (y)

# Step 4.1 : Training & Test split ( random_state 42)

In [ ]:
# training 
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Step 4.2 : Scale data

In [ ]:
# scale data 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print (X_test_scaled)

# Step 5.1 : EDA (Exploratory Data Analysis), show Distributions 


In [ ]:
# EDA
sns.countplot(x='Quality of Sleep', data=df)
plt.title("Sleep Quality Distribution")
plt.show()

# Step 5.2 : Show correlation plot

In [ ]:
# correlation 
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Correlation Between Features")
plt.show()

# Step 5.3 : Show classs implance 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

sns.countplot(x='Quality of Sleep', data=df)
plt.title("Class Distribution: Quality of Sleep")
plt.xlabel("Sleep Quality")
plt.ylabel("Count")
plt.show()

df['Quality of Sleep'].value_counts(normalize=True)


# Step 5.4: Outlier Detection

In [ ]:
num_cols = ['Age', 'Sleep Duration', 'Physical Activity Level', 'Stress Level']

for col in num_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Outlier Check: {col}")
    plt.show()

# Step 6:  Data cleaaning (this step failed with mee :))


In [ ]:
df.drop(columns=["Person ID"], inplace=True)

# Step 7 : Feature Engineering 

here my new features and why i need themm .. 


Age Group --> 	hrrelps capture non-linear effects of age 


High Stress -> 	converts continuous stress level into a clear risk indicator


Sleep Efficiency	-> shows how much sleep a person gets relative to their activity (behavioral insight)

In [ ]:
df_fe = df.copy()

# ffeature 1: Age Group 
df_fe['Age Group'] = pd.cut(df_fe['Age'], bins=[0, 25, 45, 65, 100], labels=['Youth', 'Adult', 'Mid-Age', 'Senior'])

# feature 2: iis High Stress
df_fe['High Stress'] = df_fe['Stress Level'].apply(lambda x: 1 if x >= 7 else 0)

# feature 3: ssleep Efficiency = sleep Duration / pphysical acctivity Leveel
df_fe['Sleep Efficiency'] = df_fe['Sleep Duration'] / (df_fe['Physical Activity Level'] + 1) 

#encncode new categorical feature
df_fe['Age Group'] = LabelEncoder().fit_transform(df_fe['Age Group'].astype(str))

# sshow new columns
df_fe[['Age', 'Age Group', 'Stress Level', 'High Stress', 'Sleep Duration', 'Physical Activity Level', 'Sleep Efficiency']].head()


# Step 8.1  : Prepare Your Data for ML, Set features X and target y


In [ ]:
# here with engineerd dataframe ,,
df = df_fe.copy()

# my target and feature
X = df.drop(columns=['Quality of Sleep'])  # features
y = df['Quality of Sleep']                 # target


# Step 8.2 :Split into Train / Val / Test


In [ ]:
from sklearn.model_selection import train_test_split

# first split 70% train, 30% temp ....................
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

#tthen split temp 50/50 into val and test (15% each)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Print shapes
print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")


# Step 8.3 : Scale the Features

In [ ]:
from sklearn.preprocessing import StandardScaler
# start with scalerr
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


Now myy data is:

xlean

transformed

split correctly



# Step 9 :  Train 5 Different ML Models

In [ ]:
def train_model(model, name):
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    print(f"\n{name} Performance on Validation Set:")
    print("Accuracy:", accuracy_score(y_val, y_pred))
    print("F1 Score:", f1_score(y_val, y_pred, average='weighted'))
    return model


In [ ]:
models = [
    (LogisticRegression(), "Logistic Regression"),
    (DecisionTreeClassifier(), "Decision Tree"),
    (RandomForestClassifier(), "Random Forest"),
    (GradientBoostingClassifier(), "Gradient Boosting"),
    (SVC(), "Support Vector Machine")
]

best_model = None
best_f1 = 0

for model, name in models:
    m = train_model(model, name)
    y_pred = m.predict(X_val_scaled)
    score = f1_score(y_val, y_pred, average='weighted')
    if score > best_f1:
        best_f1 = score
        best_model = m


# Step 10: Choose the Best Model + Hyperparameter Tuning,
# bboth Logistic Regression and Random Forest achieved perfect performance on the validation set (1.0 accuracy and F1 score).

In [ ]:
print("\n🏁 Final Evaluation on Test Set")
y_test_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_test_pred))


# Step 10.1: Setup Search Space


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf = RandomForestClassifier(random_state=42)

# Randomized search ..... 
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

print("Best Parameters:", random_search.best_params_)


# Step 10.2: Evaluate Tuned Model on Validation Set

In [ ]:
tuned_rf = random_search.best_estimator_

#beest model....
y_val_pred = tuned_rf.predict(X_val_scaled)

# mmetrics
from sklearn.metrics import accuracy_score, f1_score

print("\nnn tttuned Random Forest Performance:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("F1 Score:", f1_score(y_val, y_val_pred, average='weighted'))


# Step 10.3 before and after comapring 
| Metric       | Before Tuning | After Tuning |
|--------------|----------------|----------------|
| Accuracy     | 1.00           | 1.00  |
| F1 Score     | 1.00           | 1.00  |

# Step 11.1: Final Testing, predict on test set


In [ ]:
y_test_pred = tuned_rf.predict(X_test_scaled)


# Step 11.2 : showw final metrticss 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("📊 Final Model Performance on Test Set:")
print(classification_report(y_test, y_test_pred))

# cconfusion matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=tuned_rf.classes_)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix on Test Set")
plt.show()


# Bonus 1: Feature Importance Visualization

In [ ]:
#---------------------------Bounes----------------------
importances = tuned_rf.feature_importances_
feature_names = X.columns

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=feature_names)
plt.title("🔍 Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()



# Bonus 2: cconfusion matrix (already Done in Step 11 but Here Again for Clarity ......:)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=tuned_rf.classes_)

disp.plot(cmap='Blues')
plt.title("🧩 Confusion Matrix on Test Set")
plt.show()


# Bonus 3: Regression Residual Plots
will not help heree , my task is classification so regression residual plots do not apply.